In [1]:
import os
import time
import orjson
import numpy as np
import pandas as pd

from collections import defaultdict
from joblib import Parallel, delayed
from tqdm.notebook import tqdm

In [2]:
# Hyperparameters

HEATMAP_SHAPE = 20, 14
isTOUCH = True

In [3]:
league = [
    "England_Premier_League",
    "Spain_Laliga",
    "Italy_Serie_A",
    "Germany_Bundesliga",
    "France_Ligue1",
#     "England_Championship",
#     "Germany_Bundesliga_2"
]

In [4]:
%%time

x_bin = np.linspace(0, 100, HEATMAP_SHAPE[0] + 1)
y_bin = np.linspace(0, 100, HEATMAP_SHAPE[1] + 1)


def safe_parse_score(score):
    try:
        home_score, away_score = score.replace(" ", "").split(":")
        home_score = int(home_score)
        away_score = int(away_score)
        return home_score, away_score, home_score - away_score
    except Exception:
        return None, None, None


def is_starting_player(player):
    if "isFirstEleven" in player:
        return bool(player.get("isFirstEleven"))
    try:
        return "0" in player.get("stats", {}).get("ratings", {})
    except Exception:
        return False


def build_player_attribute_dict(players, attr_name):
    return {
        p["playerId"]: p.get(attr_name)
        for p in players
        if p.get("playerId") is not None
    }

def build_player_rating_dict(players):
    return {
        p["playerId"] : next(reversed(p["stats"].get("ratings", {}).values()), None)
        for p in players
        if p.get("stats") is not None and p.get("playerId") is not None
    }


def build_normalized_heatmap(xs, ys, normalizer):
    heatmap, _, _ = np.histogram2d(
        xs, ys,
        bins=[x_bin, y_bin],
        range=[[0, 100], [0, 100]]
    )
    heatmap = heatmap.astype(np.float32)
    if normalizer > 0:
        heatmap /= normalizer
    return heatmap

CPU times: total: 0 ns
Wall time: 0 ns


In [5]:
def process_json(league_name, json_path):
    with open(json_path, "rb") as f:
        data = orjson.loads(f.read())
    
    # Player Information
    home_players = data["home"]["players"]
    away_players = data["away"]["players"]
    players = home_players + away_players

    # Match Information
    player_age_dict = build_player_attribute_dict(players, "age")
    player_weight_dict = build_player_attribute_dict(players, "weight")
    player_height_dict = build_player_attribute_dict(players, "height")
    player_position_dict = build_player_attribute_dict(players, "position")
    player_rating_dict = build_player_rating_dict(players)
    
    starting_11_home = []
    substitute_home = []
    starting_11_away = []
    substitute_away = []
    for player in home_players:
        if is_starting_player(player):
            starting_11_home.append(player["playerId"])
        else:
            substitute_home.append(player["playerId"])
    if len(starting_11_home) != 11:
        print(f"Warning! There are {len(starting_11_home)} players (Home) starting the match: {json_path}")
    for player in away_players:
        if is_starting_player(player):
            starting_11_away.append(player["playerId"])
        else:
            substitute_away.append(player["playerId"])
    if len(starting_11_away) != 11:
        print(f"Warning! There are {len(starting_11_away)} players (Away) starting the match: {json_path}")

    home_players_set = set(starting_11_home + substitute_home)
    away_players_set = set(starting_11_away + substitute_away)
    home_x, home_y, away_x, away_y = [], [], [], []
    player_coord = {}

    for event in data["events"]:
        if isTOUCH and not event.get("isTouch", False):
            continue

        player = event.get("playerId", None)
        x = event.get("x", None)
        y = event.get("y", None)

        if player is None or x is None or y is None:
            continue

        if player in home_players_set:
            home_x.append(x)
            home_y.append(y)
        elif player in away_players_set:
            away_x.append(x)
            away_y.append(y)
        else:
            continue

        if player not in player_coord:
            player_coord[player] = [[x], [y]]
        else:
            player_coord[player][0].append(x)
            player_coord[player][1].append(y)
            
    home_touch, away_touch = len(home_x), len(away_x)
    home_heatmap = build_normalized_heatmap(home_x, home_y, home_touch)
    away_heatmap = build_normalized_heatmap(away_x, away_y, away_touch)

    player_heatmap = {}
    for player in home_players_set | away_players_set:
        coord = player_coord.get(player, [[], []])
        team_touch = home_touch if player in home_players_set else away_touch
        player_heatmap[player] = build_normalized_heatmap(coord[0], coord[1], team_touch)

    home_score, away_score, score_diff = safe_parse_score(data.get("score"))

    match_dict = {
        "League": league_name,
        "Match_Date": data["startTime"],
        "Home_Team": data["home"]["name"],
        "Away_Team": data["away"]["name"],
        "Home_Team_ID": data["home"]["teamId"],
        "Away_Team_ID": data["away"]["teamId"],
        "Home_Score": home_score,
        "Away_Score": away_score,
        "Score_Diff": score_diff,
        "Starting_11(Home)": starting_11_home,
        "Substitute(Home)": substitute_home,
        "Starting_11(Away)": starting_11_away,
        "Substitute(Away)": substitute_away,
        "Manager(Home)": data["home"]["managerName"],
        "Manager(Away)": data["away"]["managerName"],
        "Referee": data["referee"]["officialId"],
        "Player_Age": player_age_dict,
        "Player_Weight": player_weight_dict,
        "Player_Height": player_height_dict,
        "Player_Position": player_position_dict,
        "Player_Rating": player_rating_dict,
        "Team_Heatmap(Home)": home_heatmap,
        "Team_Heatmap(Away)": away_heatmap,
        "Player_Heatmap": player_heatmap,
    }

    return match_dict

In [6]:
%%time

tasks = []
for league_name in league:
    json_files = [
        js.path
        for js in os.scandir(league_name)
        if js.is_file() and js.name.endswith(".json")
    ]
    tasks.extend((league_name, json_path) for json_path in json_files)

results = Parallel(
    n_jobs=12,
    backend="loky",
    batch_size=5,
    verbose=10,
)(
    delayed(process_json)(league_name, json_path)
    for league_name, json_path in tasks
)

[Parallel(n_jobs=12)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done   2 tasks      | elapsed:    0.3s
[Parallel(n_jobs=12)]: Done  12 tasks      | elapsed:    0.3s
[Parallel(n_jobs=12)]: Done  29 tasks      | elapsed:    0.4s
[Parallel(n_jobs=12)]: Done  64 tasks      | elapsed:    0.4s
[Parallel(n_jobs=12)]: Done 109 tasks      | elapsed:    0.6s
[Parallel(n_jobs=12)]: Done 154 tasks      | elapsed:    0.7s
[Parallel(n_jobs=12)]: Done 209 tasks      | elapsed:    0.8s
[Parallel(n_jobs=12)]: Done 264 tasks      | elapsed:    0.9s
[Parallel(n_jobs=12)]: Done 329 tasks      | elapsed:    1.0s
[Parallel(n_jobs=12)]: Done 394 tasks      | elapsed:    1.2s
[Parallel(n_jobs=12)]: Done 469 tasks      | elapsed:    1.3s
[Parallel(n_jobs=12)]: Done 544 tasks      | elapsed:    1.5s
[Parallel(n_jobs=12)]: Done 629 tasks      | elapsed:    1.7s
[Parallel(n_jobs=12)]: Done 714 tasks      | elapsed:    1.8s
[Parallel(n_jobs=12)]: Done 809 tasks      | elapsed:  

CPU times: total: 17.9 s
Wall time: 1min 41s


[Parallel(n_jobs=12)]: Done 19714 out of 19714 | elapsed:  1.7min finished


In [7]:
match_df = pd.DataFrame(results)

In [8]:
match_df.columns

Index(['League', 'Match_Date', 'Home_Team', 'Away_Team', 'Home_Team_ID',
       'Away_Team_ID', 'Home_Score', 'Away_Score', 'Score_Diff',
       'Starting_11(Home)', 'Substitute(Home)', 'Starting_11(Away)',
       'Substitute(Away)', 'Manager(Home)', 'Manager(Away)', 'Referee',
       'Player_Age', 'Player_Weight', 'Player_Height', 'Player_Position',
       'Player_Rating', 'Team_Heatmap(Home)', 'Team_Heatmap(Away)',
       'Player_Heatmap'],
      dtype='object')

In [9]:
match_df

,League,Match_Date,Home_Team,Away_Team,Home_Team_ID,Away_Team_ID,Home_Score,Away_Score,Score_Diff,Starting_11(Home),...,Manager(Away),Referee,Player_Age,Player_Weight,Player_Height,Player_Position,Player_Rating,Team_Heatmap(Home),Team_Heatmap(Away),Player_Heatmap
0,England_Premier_League,2016-05-15T15:00:00,Arsenal,Aston Villa,13,24,4,0,4,"[6775, 125211, 76810, 30051, 23072, 69738, 452...",...,Eric Black,90,"{6775: 44, 125211: 31, 76810: 35, 30051: 40, 2...","{6775: 90, 125211: 74, 76810: 72, 30051: 75, 2...","{6775: 196, 125211: 178, 76810: 185, 30051: 18...","{6775: 'GK', 125211: 'DR', 76810: 'DC', 30051:...","{6775: 7.08, 125211: 8.38, 76810: 7.51, 30051:...","[[0.0012919897, 0.0, 0.0, 0.0012919897, 0.0, 0...","[[0.005076142, 0.0033840947, 0.0, 0.0016920473...","{26820: [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0..."
1,England_Premier_League,2015-12-28T17:30:00,Arsenal,Bournemouth,13,183,2,0,2,"[6775, 125211, 76810, 6292, 27550, 124316, 268...",...,Eddie Howe,124,"{6775: 44, 125211: 31, 76810: 35, 6292: 41, 27...","{6775: 90, 125211: 74, 76810: 72, 6292: 90, 27...","{6775: 196, 125211: 178, 76810: 185, 6292: 198...","{6775: 'GK', 125211: 'DR', 76810: 'DC', 6292: ...","{6775: 7.07, 125211: 7.58, 76810: 8.58, 6292: ...","[[0.0, 0.0, 0.0014164306, 0.004249292, 0.00283...","[[0.0, 0.0, 0.0, 0.0, 0.0015037594, 0.0, 0.001...","{81026: [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0..."
2,England_Premier_League,2016-01-24T16:00:00,Arsenal,Chelsea,13,15,0,1,-1,"[6775, 125211, 30051, 6292, 23072, 6321, 26820...",...,Guus Hiddink,90,"{6775: 44, 125211: 31, 30051: 40, 6292: 41, 23...","{6775: 90, 125211: 74, 30051: 75, 6292: 90, 23...","{6775: 196, 125211: 178, 30051: 186, 6292: 198...","{6775: 'GK', 125211: 'DR', 30051: 'DC', 6292: ...","{6775: 7.06, 125211: 7.25, 30051: 6.67, 6292: ...","[[0.0, 0.00311042, 0.0, 0.00311042, 0.0, 0.004...","[[0.0, 0.0014124294, 0.0014124294, 0.001412429...","{33404: [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0..."
3,England_Premier_League,2016-04-17T16:00:00,Arsenal,Crystal Palace,13,162,1,1,0,"[6775, 125211, 76810, 30051, 23072, 125209, 69...",...,Alan Pardew,124,"{6775: 44, 125211: 31, 76810: 35, 30051: 40, 2...","{6775: 90, 125211: 74, 76810: 72, 30051: 75, 2...","{6775: 196, 125211: 178, 76810: 185, 30051: 18...","{6775: 'GK', 125211: 'DR', 76810: 'DC', 30051:...","{6775: 6.37, 125211: 6.77, 76810: 7.76, 30051:...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0011111111, 0.003...","[[0.0, 0.0, 0.0, 0.004415011, 0.0, 0.008830022...","{21571: [[0.0, 0.0, 0.0, 0.0022075055, 0.0, 0...."
4,England_Premier_League,2015-10-24T17:30:00,Arsenal,Everton,13,31,2,1,1,"[6775, 125211, 76810, 30051, 23072, 69738, 452...",...,Roberto Martinez,83,"{6775: 44, 125211: 31, 76810: 35, 30051: 40, 2...","{6775: 90, 125211: 74, 76810: 72, 30051: 75, 2...","{6775: 196, 125211: 178, 76810: 185, 30051: 18...","{6775: 'GK', 125211: 'DR', 76810: 'DC', 30051:...","{6775: 6.52, 125211: 6.94, 76810: 6.94, 30051:...","[[0.0013661202, 0.0, 0.0013661202, 0.0, 0.0, 0...","[[0.0, 0.0, 0.0, 0.0016920473, 0.0, 0.00676818...","{131523: [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19709,France_Ligue1,2026-01-17T18:00:00,Toulouse,Nice,246,613,5,1,4,"[468899, 402059, 10454, 400646, 525757, 339671...",...,Claude Puel,3221,"{468899: 21, 402059: 29, 10454: 33, 400646: 23...","{468899: 82, 402059: 81, 10454: 71, 400646: 74...","{468899: 188, 402059: 191, 10454: 182, 400646:...","{468899: 'GK', 402059: 'DC', 10454: 'DC', 4006...","{468899: 7.42, 402059: 7.39, 10454: 6.01, 4006...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0016750419, 0.010...","[[0.0014326648, 0.0, 0.0014326648, 0.0, 0.0014...","{400646: [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.001..."
19710,France_Ligue1,2026-02-21T18:00:00,Toulouse,Paris FC,246,2832,1,1,0,"[468899, 402059, 10454, 400646, 397564, 534461...",...,Stéphane Gilli,899,"{468899: 21, 402059: 29, 10454: 33, 400646: 23...","{468899: 82, 402059: 81, 10454: 71, 400646: 74...","{468899: 188, 

In [10]:
%%time
match_df.to_pickle("dataset.pkl")

CPU times: total: 9.55 s
Wall time: 9.71 s
